In [122]:
%pip -q install yfinance pandas numpy matplotlib seaborn plotly \
    scikit-learn scipy pyarrow


In [123]:
import yfinance as yf
import pandas as pd

tickers = {
    "RELIANCE": "RELIANCE.NS",
    "HDFCBANK": "HDFCBANK.NS",
    "ICICIBANK": "ICICIBANK.NS",
    "INFY": "INFY.NS",
    "TCS": "TCS.NS",
    "LT": "LT.NS",
    "ITC": "ITC.NS",
    "SBIN": "SBIN.NS",
    "BHARTIARTL": "BHARTIARTL.NS",
    "AXISBANK": "AXISBANK.NS"
}

prices = pd.DataFrame()

for name, ticker in tickers.items():
    df = yf.download(
        ticker,
        start="2016-10-26",
        end="2026-10-27",
        auto_adjust=True,
        progress=False
    )

    if not df.empty:
        prices[name] = df["Close"].squeeze()

prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name = "Date"
prices = prices.sort_index()

print("Stocks downloaded:", len(prices.columns))
print(prices.columns.tolist())
print("Rows:", len(prices))
display(prices.head())

Stocks downloaded: 10
['RELIANCE', 'HDFCBANK', 'ICICIBANK', 'INFY', 'TCS', 'LT', 'ITC', 'SBIN', 'BHARTIARTL', 'AXISBANK']
Rows: 2471


,RELIANCE,HDFCBANK,ICICIBANK,INFY,TCS,LT,ITC,SBIN,BHARTIARTL,AXISBANK
Date,,,,,,,,,,
2016-10-26,229.904251,282.569305,239.191315,387.409363,939.617371,847.363037,160.209305,232.423676,268.897125,479.832458
2016-10-27,228.737320,285.487823,244.260406,383.322113,948.185059,843.361450,163.128326,229.414474,273.467651,477.962494
2016-10-28,230.100555,287.072449,237.859650,381.030182,940.578186,846.448425,163.128326,231.749954,269.870453,479.192719
2016-11-01,229.238998,287.243530,236.570892,377.955231,921.482178,836.415894,161.014557,232.603302,268.685516,468.268402
2016-11-02,223.295364,284.131195,233.606750,374.746490,903.464050,828.126831,159.236343,225.507095,262.041412,469.252594


In [124]:
import pandas as pd
import numpy as np

def read_market_file(filename, new_name):
    # auto-detect delimiter and strip UTF-8 BOM automatically
    df = pd.read_csv(filename, sep=None, engine='python', encoding='utf-8-sig')

    # Strip double quotes and whitespaces from the column headers
    df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

    # Identify the date and price columns safely with a default fallback
    date_col = next(
        (c for c in df.columns if c.lower() in ["date", "time"]),
        None
    )

    price_col = next(
        (c for c in df.columns if c.lower() in ["price", "close"]),
        None
    )

    if date_col is None or price_col is None:
        raise ValueError(
            f"Could not find required columns in {filename}. "
            f"Available columns are: {df.columns.tolist()}"
        )

    # Keep only date and closing price
    df = df[[date_col, price_col]].copy()
    df.columns = ["Date", new_name]

    # Convert text into proper data types
    df["Date"] = pd.to_datetime(
        df["Date"], errors="coerce", dayfirst=True
    )

    df[new_name] = pd.to_numeric(
        df[new_name].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    df = df.dropna()
    df = df.drop_duplicates("Date")
    df = df.sort_values("Date")

    return df.set_index("Date")


nifty = read_market_file("nifty50.csv", "NIFTY50")
vix = read_market_file("india_vix.csv", "VIX")

print("NIFTY 50:", nifty.shape)
print("India VIX:", vix.shape)

display(nifty.head())
display(vix.head())

NIFTY 50: (2479, 1)
India VIX: (2475, 1)


/tmp/ipykernel_1084/3751185477.py:33: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(


,NIFTY50
Date,
2016-10-26,8615.25
2016-10-27,8615.25
2016-10-28,8638.00
2016-10-30,8625.70
2016-11-01,8626.25


,VIX
Date,
2016-10-26,14.4525
2016-10-27,14.0850
2016-10-28,14.6525
2016-11-01,15.7350
2016-11-02,16.7975


In [125]:
# combine all the data

data = nifty.join(vix, how="inner")   # The inner join keeps dates available in all datasets
data = data.join(prices, how = "inner")

data = data.sort_index()
data = data.replace([np.inf, -np.inf], np.nan)

print("Combined Dataset shape : ", data.shape)
print("Date range : ", data.index.min(), data.index.max())
print("Missing values:\n", data.isna().sum())

display(data.head())
display(data.tail())


Combined Dataset shape :  (2468, 12)
Date range :  2016-10-26 00:00:00 2026-10-26 00:00:00
Missing values:
 NIFTY50       0
VIX           0
RELIANCE      0
HDFCBANK      0
ICICIBANK     0
INFY          0
TCS           0
LT            0
ITC           0
SBIN          0
BHARTIARTL    0
AXISBANK      0
dtype: int64


,NIFTY50,VIX,RELIANCE,HDFCBANK,ICICIBANK,INFY,TCS,LT,ITC,SBIN,BHARTIARTL,AXISBANK
Date,,,,,,,,,,,,
2016-10-26,8615.25,14.4525,229.904251,282.569305,239.191315,387.409363,939.617371,847.363037,160.209305,232.423676,268.897125,479.832458
2016-10-27,8615.25,14.0850,228.737320,285.487823,244.260406,383.322113,948.185059,843.361450,163.128326,229.414474,273.467651,477.962494
2016-10-28,8638.00,14.6525,230.100555,287.072449,237.859650,381.030182,940.578186,846.448425,163.128326,231.749954,269.870453,479.192719
2016-11-01,8626.25,15.7350,229.238998,287.243530,236.570892,377.955231,921.482178,836.415894,161.014557,232.603302,268.685516,468.268402
2016-11-02,8514.00,16.7975,223.295364,284.131195,233.606750,374.746490,903.464050,828.126831,159.236343,225.507095,262.041412,469.252594


,NIFTY50,VIX,RELIANCE,HDFCBANK,ICICIBANK,INFY,TCS,LT,ITC,SBIN,BHARTIARTL,AXISBANK
Date,,,,,,,,,,,,
2026-10-19,22888.95,14.7725,1202.000000,712.000000,1381.000000,1130.699951,2460.899902,3661.899902,260.799988,973.700012,1875.199951,1236.000000
2026-10-21,22811.35,15.6750,1209.000000,706.950012,1385.000000,1168.000000,2529.000000,3636.000000,262.000000,963.000000,1891.699951,1210.000000
2026-10-22,22687.65,15.7625,1184.500000,694.049988,1398.500000,1153.349976,2506.899902,3638.300049,258.899994,970.000000,1895.000000,1212.000000
2026-10-23,22600.00,16.8600,1190.000000,692.799988,1394.199951,1153.800049,2530.000000,3647.000000,257.899994,958.599976,1892.099976,1217.500000
2026-10-26,22586.80,17.5425,1195.900024,687.549988,1387.099976,1141.000000,2516.699951,3650.399902,257.500000,952.650024,1893.300049,1216.599976


In [126]:
# Daily percentage return
data["Return"] = data["NIFTY50"].pct_change()

# Volatility over the previous 20 trading sessions
data["Volatility"] = data["Return"].rolling(20).std()

# Momentum over 20 trading sessions
data["Momentum"] = data["NIFTY50"].pct_change(20)

# Drawdown from the previous highest index level
data["Drawdown"] = (
    data["NIFTY50"] / data["NIFTY50"].cummax()
) - 1

# Remove rows where rolling calculations are unavailable
data = data.dropna().copy()

features = [
    "Return",
    "Volatility",
    "Momentum",
    "VIX",
    "Drawdown"
]

print("Features created:", features)
display(data[features].head())

Features created: ['Return', 'Volatility', 'Momentum', 'VIX', 'Drawdown']


,Return,Volatility,Momentum,VIX,Drawdown
Date,,,,,
2016-11-25,0.018681,0.011542,-0.058147,17.6125,-0.060627
2016-11-28,0.001553,0.011568,-0.056684,18.0550,-0.059169
2016-11-29,0.001876,0.011550,-0.057403,17.7075,-0.057403
2016-11-30,0.010114,0.011909,-0.046573,16.8500,-0.047870
2016-12-01,-0.003842,0.011649,-0.037714,16.6875,-0.051528


In [127]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select the input features directly from the 'data' DataFrame
X = data[features].copy()

# Standardize the features so that different scales do not dominate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train K-Means
model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

data["Cluster"] = model.fit_predict(X_scaled)

print("K-Means training completed successfully!")
print(data["Cluster"].value_counts().sort_index())

K-Means training completed successfully!
Cluster
0    1505
1      49
2     894
Name: count, dtype: int64


In [128]:
# gove the clulster understandbale names


# Compare the average volatility in each cluster
cluster_risk = data.groupby("Cluster")["Volatility"].mean()

# Rank clusters from lowest to highest volatility
ordered = cluster_risk.sort_values().index.tolist()

names = {
    ordered[0]: "Low Volatility",
    ordered[1]: "Medium Volatility",
    ordered[2]: "High Volatility"
}

data["Regime"] = data["Cluster"].map(names)

print(cluster_risk)
display(data[["NIFTY50", "VIX", "Volatility", "Regime"]].tail())

Cluster
0    0.006993
1    0.036201
2    0.010077
Name: Volatility, dtype: float64


,NIFTY50,VIX,Volatility,Regime
Date,,,,
2026-10-19,22888.95,14.7725,0.010375,Medium Volatility
2026-10-21,22811.35,15.6750,0.010348,Medium Volatility
2026-10-22,22687.65,15.7625,0.010377,Medium Volatility
2026-10-23,22600.00,16.8600,0.010279,Medium Volatility
2026-10-26,22586.80,17.5425,0.009690,Medium Volatility


In [133]:
# Chart A — NIFTY 50 with regime colors

import plotly.express as px

fig = px.scatter(
    data.reset_index(),
    x="Date",
    y="NIFTY50",
    color="Regime",
    title="NIFTY 50: Machine Learning-Based Market Regimes",
    labels={"NIFTY50": "NIFTY 50 Index Level"},
    opacity=0.7
)

fig.update_traces(marker_size=4)
fig.show()


# This displays the index's historical movement, with each observation colored according to the regime assigned by K-Means.

In [135]:
regime_summary = data.groupby("Regime").agg(
    Average_Daily_Return=("Return", "mean"),
    Average_Volatility=("Volatility", "mean"),
    Average_VIX=("VIX", "mean"),
    Average_Drawdown=("Drawdown", "mean"),
    Trading_Days=("Regime", "size")
)

regime_summary["Average_Daily_Return"] *= 100
regime_summary["Average_Volatility"] *= 100
regime_summary["Average_Drawdown"] *= 100

display(regime_summary.round(3))

,Average_Daily_Return,Average_Volatility,Average_VIX,Average_Drawdown,Trading_Days
Regime,,,,,
High Volatility,-0.161,3.620,48.604,-27.393,49
Low Volatility,0.237,0.699,14.186,-2.177,1505
Medium Volatility,-0.260,1.008,18.409,-8.668,894


In [138]:
# Chart C — Visualize volatility by regime

fig = px.bar(
    regime_summary.reset_index(),
    x="Regime",
    y="Average_Volatility",
    color="Regime",
    title="Average Rolling Volatility by Market Regime",
    labels={"Average_Volatility": "Volatility (%)"}
)

fig.show()


In [139]:
# Chart D — PCA visualization

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

plot_data = pd.DataFrame(
    components,
    columns=["PC1", "PC2"],
    index=data.index
)

plot_data["Regime"] = data["Regime"]

fig = px.scatter(
    plot_data.reset_index(),
    x="PC1",
    y="PC2",
    color="Regime",
    title="PCA Visualization of Market Regimes",
    opacity=0.6
)

fig.show()

print(
    "Variance explained by two components:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)


Variance explained by two components: 76.59 %


In [140]:
# 7. Portfolio risk analytics

stock_names = list(stocks.keys())

# Daily returns of the 10 selected stocks
stock_returns = data[stock_names].pct_change()

# Equal-weighted portfolio: 10% allocation per stock
data["Portfolio_Return"] = stock_returns.mean(axis=1)

# Remove the first row, where daily returns are unavailable
portfolio_data = data.dropna(
    subset=["Portfolio_Return"]
).copy()

display(portfolio_data[
    ["Portfolio_Return", "Regime"]
].head())

,Portfolio_Return,Regime
Date,,
2016-11-28,0.001073,Medium Volatility
2016-11-29,-0.002677,Medium Volatility
2016-11-30,0.012122,Medium Volatility
2016-12-01,-0.005367,Medium Volatility
2016-12-02,-0.009662,Medium Volatility


In [141]:
def sharpe_ratio(returns):
    returns = returns.dropna()

    if returns.std() == 0 or len(returns) < 2:
        return np.nan

    # Assumes a zero risk-free rate
    return (
        returns.mean() / returns.std()
    ) * np.sqrt(252)


def max_drawdown(returns):
    wealth = (1 + returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    return drawdown.min()


risk_results = []

for regime, group in portfolio_data.groupby("Regime"):
    r = group["Portfolio_Return"].dropna()

    risk_results.append({
        "Regime": regime,
        "Trading_Days": len(r),
        "Annualized_Return_%": r.mean() * 252 * 100,
        "Annualized_Volatility_%": r.std() * np.sqrt(252) * 100,
        "Sharpe_Ratio": sharpe_ratio(r),
        "Maximum_Drawdown_%": max_drawdown(r) * 100,
        "VaR_95_%": r.quantile(0.05) * 100,
        "CVaR_95_%": r[r <= r.quantile(0.05)].mean() * 100
    })

risk_table = pd.DataFrame(risk_results)

display(risk_table.round(3))

,Regime,Trading_Days,Annualized_Return_%,Annualized_Volatility_%,Sharpe_Ratio,Maximum_Drawdown_%,VaR_95_%,CVaR_95_%
0,High Volatility,49,-30.924,64.541,-0.479,-23.906,-6.956,-10.568
1,Low Volatility,1505,65.682,12.218,5.376,-3.834,-0.885,-1.161
2,Medium Volatility,893,-61.856,17.518,-3.531,-90.005,-1.968,-2.744


In [142]:
# 8. Basic model validation

from sklearn.metrics import silhouette_score

score = silhouette_score(
    X_scaled,
    data["Cluster"],
    sample_size=min(2000, len(data)),
    random_state=42
)

print("Silhouette Score:", round(score, 4))

Silhouette Score: 0.2933


In [143]:
# results :

# Save the main processed dataset and analysis tables
data.to_csv("market_regimes_results.csv")
regime_summary.to_csv("regime_summary.csv")
risk_table.to_csv("portfolio_risk_by_regime.csv")

print("Files created successfully:")
print("- market_regimes_results.csv")
print("- regime_summary.csv")
print("- portfolio_risk_by_regime.csv")

Files created successfully:
- market_regimes_results.csv
- regime_summary.csv
- portfolio_risk_by_regime.csv


In [145]:
import pandas as pd
import numpy as np

results = pd.read_csv("market_regimes_results.csv")
regimes = pd.read_csv("regime_summary.csv")
risk = pd.read_csv("portfolio_risk_by_regime.csv")

print("Dataset shape:", results.shape)
print("\nRegime summary:")
display(regimes)

print("\nPortfolio risk analysis:")
display(risk)

print("\nMissing values in results:", results.isna().sum().sum())
print("Detected regimes:", results["Regime"].nunique())
print("Rows per regime:")
print(results["Regime"].value_counts())

Dataset shape: (2448, 20)

Regime summary:


,Regime,Average_Daily_Return,Average_Volatility,Average_VIX,Average_Drawdown,Trading_Days
0,High Volatility,-0.160758,3.620065,48.604286,-27.392887,49
1,Low Volatility,0.237278,0.699270,14.185972,-2.177315,1505
2,Medium Volatility,-0.259649,1.007714,18.408635,-8.668185,894



Portfolio risk analysis:


,Unnamed: 0,Regime,Trading_Days,Annualized_Return_%,Annualized_Volatility_%,Sharpe_Ratio,Maximum_Drawdown_%,VaR_95_%,CVaR_95_%
0,0,High Volatility,49,-30.924008,64.540879,-0.479138,-23.906478,-6.955853,-10.567938
1,1,Low Volatility,1505,65.681698,12.217578,5.376000,-3.833607,-0.885182,-1.160556
2,2,Medium Volatility,893,-61.855502,17.518075,-3.530953,-90.005267,-1.968150,-2.743595



Missing values in results: 1
Detected regimes: 3
Rows per regime:
Regime
Low Volatility       1505
Medium Volatility     894
High Volatility        49
Name: count, dtype: int64
